In [1]:
import geopandas as gpd

In [2]:
# Load cleaned biological data
birds_gdf = gpd.read_parquet("pilot_bird_occurrences_clean.parquet")

In [3]:
# Inspect the data to ensure the geometry column is intact
print(birds_gdf.head())

  dataResourceUid                                   dataResourceName images  \
0          dr1411                              iNaturalist Australia   None   
4         dr29617     Xeno-canto - Bird sounds from around the world   None   
5         dr29617     Xeno-canto - Bird sounds from around the world   None   
6         dr29617     Xeno-canto - Bird sounds from around the world   None   
8           dr341  Australian National Wildlife Collection provid...   None   

       dcterms:modified dcterms:language     dcterms:license   rightsHolder  \
0  2026-08-07T05:43:07Z             None  CC-BY-NC 4.0 (Int)  Marie Tarrant   
4                  None             None  CC-BY-NC 4.0 (Int)  Nigel Jackett   
5                  None             None  CC-BY-NC 4.0 (Int)  Nigel Jackett   
6                  None             None  CC-BY-NC 4.0 (Int)  Nigel Jackett   
8                  None             None               CC-BY           None   

   dcterms:accessRights dcterms:bibliographicCitat

In [4]:
# Verify the CRS 
print(f"Biological Data CRS: {birds_gdf.crs}")

Biological Data CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "GDA94 / Australian Albers", "base_crs": {"name": "GDA94", "datum": {"type": "GeodeticReferenceFrame", "name": "Geocentric Datum of Australia 1994", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4283}}, "conversion": {"name": "Australian Albers", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 0, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": 132, "unit": "degree", "id": {"authorit

In [5]:
import rasterio

In [6]:
# Extract the geometries into a list of (x, y) coordinate tuples
coords = [(x, y) for x, y in zip(birds_gdf.geometry.x, birds_gdf.geometry.y)]

# Define a helper function to sample the massive rasters efficiently
def sample_raster(raster_path, coordinate_list):
    with rasterio.open(raster_path) as src:
        # Yield an array for each point and extract the first pixel value
        return [val[0] for val in src.sample(coordinate_list)]

print("Extracting Elevation...")
birds_gdf['elevation'] = sample_raster("pilot_elevation_cropped.tif", coords)

print("Extracting Slope...")
birds_gdf['slope'] = sample_raster("pilot_dem_slope.tif", coords)

print("Extracting Aspect...")
birds_gdf['aspect'] = sample_raster("pilot_dem_aspect.tif", coords)

Extracting Elevation...
Extracting Slope...
Extracting Aspect...


In [7]:
# Verify the newly appended terrain columns
print(birds_gdf[['geometry', 'elevation', 'slope', 'aspect']].head())

                            geometry   elevation      slope      aspect
0     POINT (937062.86 -2602304.807)  113.804688  12.086203  229.215256
4   POINT (-975330.061 -2762410.016)  524.994629  23.021828   28.830984
5   POINT (-975330.061 -2762410.016)  524.994629  23.021828   28.830984
6   POINT (-975330.061 -2762410.016)  524.994629  23.021828   28.830984
8  POINT (-1131975.559 -2573076.306)  537.424866  18.381567    0.392016


In [8]:
import numpy as np
import pandas as pd

In [9]:
# Format the presence df
presence_df = pd.DataFrame({
    'x_coord': birds_gdf.geometry.x,
    'y_coord': birds_gdf.geometry.y,
    'presence': 1,
    'elevation': birds_gdf['elevation'],
    'slope': birds_gdf['slope'],
    'aspect': birds_gdf['aspect']
}).dropna()

num_presences = len(presence_df)

In [10]:
# Extract bounding box from the elevation raster to bound random coordinate generation
with rasterio.open("pilot_elevation_cropped.tif") as src:
    minx, miny, maxx, maxy = src.bounds

# Generate candidate pseudo-absence coordinates (oversampling to allow for NaN drops)
np.random.seed(42)
rand_x = np.random.uniform(minx, maxx, num_presences * 3)
rand_y = np.random.uniform(miny, maxy, num_presences * 3)
absence_coords = list(zip(rand_x, rand_y))

In [11]:
# Extract environmental covariates at the generated pseudo-absence locations
print("Sampling background terrain for pseudo-absences...")
absence_df = pd.DataFrame({
    'x_coord': rand_x,
    'y_coord': rand_y,
    'presence': 0,
    'elevation': sample_raster("pilot_elevation_cropped.tif", absence_coords),
    'slope': sample_raster("pilot_dem_slope.tif", absence_coords),
    'aspect': sample_raster("pilot_dem_aspect.tif", absence_coords)
})

# Drop nodata/NaN values and downsample to achieve an exact 1:1 balance
absence_df = absence_df.dropna().sample(n=num_presences, random_state=42)

Sampling background terrain for pseudo-absences...


In [12]:
# Concatenate presence and absence data and export
training_matrix = pd.concat([presence_df, absence_df], ignore_index=True)
training_matrix.to_csv("training_matrix_with_absences.csv", index=False)

print(f"Exported {len(training_matrix)} balanced records ({num_presences} presences, {num_presences} absences) to training_matrix_with_absences.csv")

Exported 110 balanced records (55 presences, 55 absences) to training_matrix_with_absences.csv
